# Phase 2 - Synthetic 3D GIS workflow

> **Safety notice:** Every feature in this notebook is fictional and non-authoritative. The scene is georeferenced near St Lucia only to test coordinate handling; it does not show real UQ buildings or utilities.

This notebook generates terrain, LoD1 and LoD2 buildings, three underground utility networks, coordinate-preserving GIS files, fixed-camera PNGs and an interactive PyVista scene.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'pyproject.toml').exists():
    project_root = project_root.parent
if not (project_root / 'pyproject.toml').exists():
    raise RuntimeError('Could not locate the project root containing pyproject.toml')

src_path = str(project_root / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from IPython.display import JSON, Image, display

from uqgems.phase2 import run_phase2
from uqgems.scene import build_plotter
from uqgems.synthetic import create_synthetic_dataset

project_root

## Generate and validate every output

The pipeline is deliberately rerunnable: all generated files are replaced from the same deterministic source definitions.

In [ ]:
summary = run_phase2(project_root)
display(JSON(summary, expanded=False))
print(f"Phase 2 status: {summary['status']}")

## Fixed-camera visual checks

These views make visual regressions easier to compare than a freely moved camera. The transparent underground view checks that utility elevations remain below terrain.

In [ ]:
for output_name in ('plan_png', 'oblique_png', 'underground_png'):
    image_path = project_root / summary['outputs'][output_name]
    print(image_path.name)
    display(Image(filename=str(image_path), width=900))

## Interactive PyVista view

Drag to orbit, scroll to zoom and use the toolbar to reset the camera. The HTML notebook backend embeds the scene so it remains interactive in VS Code without opening QGIS.

In [ ]:
dataset = create_synthetic_dataset()
interactive_plotter = build_plotter(
    dataset,
    camera='oblique',
    terrain_opacity=0.55,
    off_screen=True,
    notebook=True,
)
interactive_viewer = interactive_plotter.show(
    jupyter_backend='html', return_viewer=True
)
interactive_viewer

## Decision gate

Phase 3 should begin only when every automated check below passes and the three fixed views look sensible.

In [ ]:
failed_checks = [name for name, passed in summary['checks'].items() if not passed]
assert not failed_checks, failed_checks
assert summary['status'] == 'passed'
print(f"Decision gate passed: {len(summary['checks'])} checks succeeded.")